In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

project_path = Path.home() / "Documents" / "17th_Lok_Sabha_Analysis"
data_path = project_path / "data" / "processed" / "17th_cleaned.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (559, 97)


,Name,Constituency,Minister,Attendance,Debates,Questions,Private Member Bills,State,Party,Nature of membership,...,Debates (Clarification On The Statement By Minister),Debates (Felicitations),Questions (Consumer Affairs),Questions (Micro),Debates (Ls - Half-An-Hour Discussions (Rule-55)),Questions (Atomic Energy),Questions (Heavy Industries),Questions (Space),"Questions (Personnel, Public Grievances And Pensions)",End of Term Date
0,Abdul Khaleque,Barpeta,No,92.00%,68,279,8,Assam,Indian National Congress,Elected,...,0,0,0,0,0,0,0,0,0,NaN
1,Abhishek Banerjee,Diamond Harbour,No,19.80%,2,84,0,West Bengal,All India Trinamool Congress,Elected,...,0,0,0,0,0,0,0,0,0,NaN
2,Abu Hasem Khan Choudhury,Maldaha Dakshin,No,55.93%,5,12,0,West Bengal,Indian National Congress,Elected,...,0,0,0,0,0,0,0,0,0,NaN
3,Abu Taher Khan,Murshidabad,No,63.07%,6,4,0,West Bengal,All India Trinamool Congress,Elected,...,0,0,0,0,0,0,0,0,0,NaN
4,Achyutananda Samanta,Kandhamal,No,65.47%,33,169,0,Odisha,Biju Janata Dal,Elected,...,0,0,0,0,0,0,0,0,0,NaN


In [3]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())

Rows: 559
Columns: 97
Duplicate rows: 0


In [4]:
print("Data types:")
display(df.dtypes.value_counts())

print("\nMissing values in important columns:")
display(
    df[
        [
            "Name",
            "Constituency",
            "Minister",
            "Attendance",
            "Debates",
            "Questions",
            "Private Member Bills",
            "Party",
            "State"
        ]
    ].isnull().sum()
)

print("\nAttendance data type:", df["Attendance"].dtype)
print("Sample attendance values:", df["Attendance"].head().tolist())

Data types:


int64      81
str        15
float64     1
Name: count, dtype: int64


Missing values in important columns:


Name                     0
Constituency             0
Minister                 0
Attendance              34
Debates                  0
Questions                0
Private Member Bills     0
Party                    0
State                    0
dtype: int64


Attendance data type: str
Sample attendance values: ['92.00%', '19.80%', '55.93%', '63.07%', '65.47%']


In [5]:
# Keep the original values, but create a numeric attendance column
df["Attendance_pct"] = pd.to_numeric(
    df["Attendance"]
        .str.replace("%", "", regex=False),
    errors="coerce"
)

print("Attendance data type:", df["Attendance_pct"].dtype)
print("Missing attendance values:", df["Attendance_pct"].isna().sum())

display(
    df[["Name", "Attendance", "Attendance_pct"]].head()
)

Attendance data type: float64
Missing attendance values: 34


,Name,Attendance,Attendance_pct
0,Abdul Khaleque,92.00%,92.00
1,Abhishek Banerjee,19.80%,19.80
2,Abu Hasem Khan Choudhury,55.93%,55.93
3,Abu Taher Khan,63.07%,63.07
4,Achyutananda Samanta,65.47%,65.47


In [6]:
display(df["Attendance_pct"].describe())

count    525.000000
mean      79.167010
std       17.264025
min        0.000000
25%       71.670000
50%       83.270000
75%       91.200000
max      100.000000
Name: Attendance_pct, dtype: float64

In [7]:
def find_mp(name):
    name = name.strip()

    if not name:
        print("Enter an MP's name.")
        return

    matches = df[
        df["Name"].str.contains(
            name,
            case=False,
            na=False,
            regex=False
        )
    ]

    if matches.empty:
        print(f"No MP found matching '{name}'.")
        return

    columns = [
        "Name",
        "Constituency",
        "Party",
        "State",
        "Minister",
        "Attendance",
        "Debates",
        "Questions",
        "Private Member Bills"
    ]

    display(matches[columns].reset_index(drop=True))

In [8]:
find_mp("Abhishek Banerjee")
find_mp("Khaleque")
find_mp("rahul")
find_mp("nonexistentperson")

,Name,Constituency,Party,State,Minister,Attendance,Debates,Questions,Private Member Bills
0,Abhishek Banerjee,Diamond Harbour,All India Trinamool Congress,West Bengal,No,19.80%,2,84,0


,Name,Constituency,Party,State,Minister,Attendance,Debates,Questions,Private Member Bills
0,Abdul Khaleque,Barpeta,Indian National Congress,Assam,No,92.00%,68,279,8


,Name,Constituency,Party,State,Minister,Attendance,Debates,Questions,Private Member Bills
0,Rahul Gandhi,Wayanad,Indian National Congress,Kerala,No,50.53%,8,99,0
1,Rahul Kaswan,Churu,Bharatiya Janata Party,Rajasthan,No,96.13%,98,277,3
2,Rahul Ramesh Shewale,Mumbai-South-Central,Shiv Sena,Maharashtra,No,95.07%,143,531,13


No MP found matching 'nonexistentperson'.


In [9]:
def filter_mps(state=None, party=None, minister=None):
    results = df.copy()

    if state:
        results = results[
            results["State"].str.contains(
                state, case=False, na=False, regex=False
            )
        ]

    if party:
        results = results[
            results["Party"].str.contains(
                party, case=False, na=False, regex=False
            )
        ]

    if minister is not None:
        results = results[
            results["Minister"].str.casefold() == minister.casefold()
        ]

    columns = [
        "Name", "Constituency", "Party", "State",
        "Minister", "Attendance", "Debates",
        "Questions", "Private Member Bills"
    ]

    display(results[columns].reset_index(drop=True))
    print(f"Matching MPs: {len(results)}")

In [10]:
filter_mps(state="Tamil Nadu")
filter_mps(party="Indian National Congress")
filter_mps(minister="Yes")
filter_mps(state="Tamil Nadu", minister="No")

,Name,Constituency,Party,State,Minister,Attendance,Debates,Questions,Private Member Bills
0,A.K.P. Chinraj,Namakkal,Dravida Munnetra Kazhagam,Tamil Nadu,No,74.07%,12,263,0
1,Andimuthu Raja,Nilgiris,Dravida Munnetra Kazhagam,Tamil Nadu,No,74.07%,43,271,2
2,A. Chellakumar,Krishnagiri,Indian National Congress,Tamil Nadu,No,66.00%,27,195,0
3,C.N. Annadurai,Tiruvannamalai,Dravida Munnetra Kazhagam,Tamil Nadu,No,80.33%,55,451,1
4,D. Ravikumar,Viluppuram,Dravida Munnetra Kazhagam,Tamil Nadu,No,68.00%,60,220,5
5,Dayanidhi Maran,Chennai Central,Dravida Munnetra Kazhagam,Tamil Nadu,No,75.73%,20,136,0
6,Dhanush M Kumar,Tenkasi,Dravida Munnetra Kazhagam,Tamil Nadu,No,98.80%,121,490,2
7,D.M. Kathir Anand,Vellore,Dravida Munnetra Kazhagam,Tamil Nadu,No,67.36%,32,262,5
8,DNV Senthilkumar S.,Dharmapuri,Dravida Munnetra Kazhagam,Tamil Nadu,No,98.80%,316,511,6
9,G. Selvam,Kancheepuram,Dravida Munnetra Kazhagam,Tamil Nadu,No,82.00%,18,436,1


Matching MPs: 40


,Name,Constituency,Party,State,Minister,Attendance,Debates,Questions,Private Member Bills
0,Abdul Khaleque,Barpeta,Indian National Congress,Assam,No,92.00%,68,279,8
1,Abu Hasem Khan Choudhury,Maldaha Dakshin,Indian National Congress,West Bengal,No,55.93%,5,12,0
2,Adhir Ranjan Chowdhury,Baharampur,Indian National Congress,West Bengal,No,89.33%,233,69,5
3,Adoor Prakash,Attingal,Indian National Congress,Kerala,No,82.87%,66,388,0
4,Amar Singh,Fatehgarh Sahib,Indian National Congress,Punjab,No,86.33%,56,217,0
5,Anto Antony,Pathanamthitta,Indian National Congress,Kerala,No,82.40%,61,386,2
6,Anumula Revanth Reddy,Malkajgiri,Indian National Congress,Telangana,No,67.07%,17,197,0
7,Balubhau Dhannorkar,Chandrapur,Indian National Congress,Maharashtra,No,67.27%,37,281,0
8,Benny Behanan,Chalakudy,Indian National Congress,Kerala,No,85.13%,69,343,3
9,A. Chellakumar,Krishnagiri,Indian National Congress,Tamil Nadu,No,66.00%,27,195,0


Matching MPs: 54


,Name,Constituency,Party,State,Minister,Attendance,Debates,Questions,Private Member Bills
0,Ajay Bhatt,Nainital-Udhamsingh Nagar,Bharatiya Janata Party,Uttarakhand,Yes,98.60%,48,45,3
1,Ajay Misra Teni,Kheri,Bharatiya Janata Party,Uttar Pradesh,Yes,94.40%,56,160,4
2,Amit Anil Chandra Shah,Gandhinagar,Bharatiya Janata Party,Gujarat,Yes,NaN,0,0,0
3,Annpurna Devi,Kodarma,Bharatiya Janata Party,Jharkhand,Yes,77.60%,31,97,0
4,Anupriya Singh Patel,Mirzapur,Apna Dal,Uttar Pradesh,Yes,96.60%,44,9,0
...,...,...,...,...,...,...,...,...,...
66,S.P. Singh Baghel,Agra,Bharatiya Janata Party,Uttar Pradesh,Yes,93.40%,23,5,0
67,Subhas Sarkar,Bankura,Bharatiya Janata Party,West Bengal,Yes,82.60%,27,52,0
68,Suresh Channabasappa Angadi,Belgaum,Bharatiya Janata Party,Karnataka,Yes,NaN,0,0,0
69,Vijay Kumar Singh,Ghaziabad,Bharatiya Janata Party,Uttar Pradesh,Yes,NaN,0,0,0


Matching MPs: 71


,Name,Constituency,Party,State,Minister,Attendance,Debates,Questions,Private Member Bills
0,A.K.P. Chinraj,Namakkal,Dravida Munnetra Kazhagam,Tamil Nadu,No,74.07%,12,263,0
1,Andimuthu Raja,Nilgiris,Dravida Munnetra Kazhagam,Tamil Nadu,No,74.07%,43,271,2
2,A. Chellakumar,Krishnagiri,Indian National Congress,Tamil Nadu,No,66.00%,27,195,0
3,C.N. Annadurai,Tiruvannamalai,Dravida Munnetra Kazhagam,Tamil Nadu,No,80.33%,55,451,1
4,D. Ravikumar,Viluppuram,Dravida Munnetra Kazhagam,Tamil Nadu,No,68.00%,60,220,5
5,Dayanidhi Maran,Chennai Central,Dravida Munnetra Kazhagam,Tamil Nadu,No,75.73%,20,136,0
6,Dhanush M Kumar,Tenkasi,Dravida Munnetra Kazhagam,Tamil Nadu,No,98.80%,121,490,2
7,D.M. Kathir Anand,Vellore,Dravida Munnetra Kazhagam,Tamil Nadu,No,67.36%,32,262,5
8,DNV Senthilkumar S.,Dharmapuri,Dravida Munnetra Kazhagam,Tamil Nadu,No,98.80%,316,511,6
9,G. Selvam,Kancheepuram,Dravida Munnetra Kazhagam,Tamil Nadu,No,82.00%,18,436,1


Matching MPs: 40


In [11]:
%pip install pymupdf

Note: you may need to restart the kernel to use updated packages.


In [12]:
from pathlib import Path
import fitz  # PyMuPDF

pdf_path = Path("data/raw/Floor.pdf")

doc = fitz.open(pdf_path)

print("Total pages:", len(doc))
print("Page 60 preview:")
print(doc[59].get_text()[:2000])

Total pages: 89
Page 60 preview:
31.01.2024 
 
58 
 
Honorable Members, 
This is my first address in this new Parliament building. This magnificent 
building has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is 
imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to 
India’s civilization and culture. It also resonates with the resolve to respect our 
democratic and parliamentary traditions. Moreover, it embodies the commitment 
to forge new traditions for the new India of the 21st century. I am confident that 
this new building will be witness to productive dialogue on policies -- policies that 
will shape the development of ‘Viksit Bharat’ in the Amrit Kaal of our 
independence. I extend my best wishes to all of you. 
Honorable Members, this year is also the 75th year of adoption of our 
Constitution. During this period, Amrit Mahotsav, the celebration of 75 years 
of independence, was completed. During this period, many programmes were 
organ

In [13]:
import pymupdf

doc = pymupdf.open(pdf_path)

for page_num in range(82, 89):
    print(f"\n{'=' * 20} PDF PAGE {page_num + 1} {'=' * 20}\n")
    print(doc[page_num].get_text()[:1200])


==================== PDF PAGE 83 ====================

31.01.2024 
 
81 
 
industry. In the last 10 years, India has successfully organized international sports 
events related to many sports. ‘Mera Yuva Bharat’ Organization has been formed 
to motivate our youth to contribute for building a ‘Viksit Bharat’ and to instill 
among them a sense of duty and spirit of service. So far, about 1 crore youth 
have joined this initiative. 
Honorable Members, we have seen the advantage of having a strong 
government during a period of upheaval. The world has been in a turmoil in the 
last 3 years. There are many conflicts raging in different parts of the world. My 
government has established India as a Vishwa-Mitra in these difficult times. It is 
because of India’s role as Vishwa-Mitra that we  have become the voice of the 
Global South today. In the last 10 years, another conventional way of thinking has 
been changed. Earlier, events related to diplomacy were confined to the corridors 
of Del

In [14]:
for page_num in [38, 58, 59, 60, 86]:
    text = doc[page_num].get_text()

    print(f"\n--- PDF PAGE {page_num + 1} ---")
    print(text[:500])


--- PDF PAGE 39 ---
31.01.2024 
 
37 
 
माननीय अÅय±: आइटम नं. 1, महासिचव । 
12.58 hrs 
PRESIDENT’S ADDRESS 
 
SECRETARY-GENERAL: I beg to lay on the Table a copy of the President’s 
Address to both Houses of Parliament assembled together on the 31st of January, 
2024. 
माननीय सदÖयगण, 
इस नए संसद भवन म¤ यह मेरा पहला संबोधन है। आज़ादी के अमृतकाल कì शुŁआत म¤ यह 
भÓय भवन बना है। यहां एक भारत ®ेķ भारत कì महक भी है।भारत कì सËयता और संÖकृित कì 
चेतना भी है। इसम¤, हमारी लोकतांिýक और संसदीय परंपराओं के सÌमान का ÿण भी है।स

--- PDF PAGE 59 ---
31.01.2024 
 
57 
 
लेिकन हमारी िवरासत ऐसी होनी चािहए िक तब कì पीढ़ी हम¤ याद करे। 
आप सभी को बहòत-बहòत शुभकामनाएं! 
धÆयवाद!  
जय िहÆद! 
जय भारत! 
 
 
 


--- PDF PAGE 60 ---
31.01.2024 
 
58 
 
Honorable Members, 
This is my first address in this new Parliament building. This magnificent 
building has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is 
imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to 
India’s

In [15]:
for page_num in [38, 58, 59, 60, 86]:
    text = doc[page_num].get_text()

    replacement_chars = text.count("\ufffd")
    mojibake_markers = sum(
        text.count(marker)
        for marker in ["Å", "Ö", "¤", "æ", "ÿ", "þ", "®"]
    )

    print(
        f"PDF page {page_num + 1}: "
        f"{len(text)} characters | "
        f"replacement chars: {replacement_chars} | "
        f"possible mojibake markers: {mojibake_markers}"
    )

PDF page 39: 1634 characters | replacement chars: 0 | possible mojibake markers: 18
PDF page 59: 151 characters | replacement chars: 0 | possible mojibake markers: 1
PDF page 60: 1597 characters | replacement chars: 0 | possible mojibake markers: 0
PDF page 61: 1671 characters | replacement chars: 0 | possible mojibake markers: 0
PDF page 87: 1072 characters | replacement chars: 0 | possible mojibake markers: 29


In [16]:
for page_num in [38, 48, 58]:
    print(f"\n{'=' * 15} PDF PAGE {page_num + 1} {'=' * 15}\n")
    print(doc[page_num].get_text()[:1000])


=============== PDF PAGE 39 ===============

31.01.2024 
 
37 
 
माननीय अÅय±: आइटम नं. 1, महासिचव । 
12.58 hrs 
PRESIDENT’S ADDRESS 
 
SECRETARY-GENERAL: I beg to lay on the Table a copy of the President’s 
Address to both Houses of Parliament assembled together on the 31st of January, 
2024. 
माननीय सदÖयगण, 
इस नए संसद भवन म¤ यह मेरा पहला संबोधन है। आज़ादी के अमृतकाल कì शुŁआत म¤ यह 
भÓय भवन बना है। यहां एक भारत ®ेķ भारत कì महक भी है।भारत कì सËयता और संÖकृित कì 
चेतना भी है। इसम¤, हमारी लोकतांिýक और संसदीय परंपराओं के सÌमान का ÿण भी है।साथ ही, 
21वé सदी के नए भारत के िलए, नई परंपराओं के िनमाªण का संकÐप भी है।मुझे पूरा िवĵास है िक इस 
नए भवन म¤ नीितयŌ पर साथªक संवाद होगा। ऐसी नीितयां जो आज़ादी के अमृतकाल म¤ िवकिसत 
भारत का िनमाªण कर¤गी। म§ आप सभी को अपनी शुभकामनाएं देती हóं। 
माननीय सदÖयगण, यह हमारे संिवधान के लागू होने का भी 75वां वषª है । इसी कालखंड म¤ 
आज़ादी के 75 वषª का उÂसव, अमृत महोÂसव भी संपÆन हòआ है । इस दौरान देश भर म¤ अनेक 
कायªøम हòए । देश ने अपने गुमनाम Öवतंýता सेनािनयŌ को

In [17]:
for page_num in [38, 48, 58, 59, 60, 86]:
    print(f"\n--- PDF PAGE {page_num + 1} ---")
    for font in doc[page_num].get_fonts(full=True):
        print(font)


--- PDF PAGE 39 ---
(11, 'ttf', 'Type0', 'CIDFont+F1', 'F1', 'Identity-H', 0)
(19, 'ttf', 'Type0', 'CIDFont+F2', 'F2', 'Identity-H', 0)
(174, 'ttf', 'Type0', 'CIDFont+F7', 'F3', 'Identity-H', 0)
(182, 'ttf', 'Type0', 'CIDFont+F8', 'F4', 'Identity-H', 0)

--- PDF PAGE 49 ---
(11, 'ttf', 'Type0', 'CIDFont+F1', 'F1', 'Identity-H', 0)

--- PDF PAGE 59 ---
(11, 'ttf', 'Type0', 'CIDFont+F1', 'F1', 'Identity-H', 0)
(160, 'ttf', 'Type0', 'CIDFont+F6', 'F2', 'Identity-H', 0)

--- PDF PAGE 60 ---
(11, 'ttf', 'Type0', 'CIDFont+F1', 'F1', 'Identity-H', 0)

--- PDF PAGE 61 ---
(11, 'ttf', 'Type0', 'CIDFont+F1', 'F1', 'Identity-H', 0)

--- PDF PAGE 87 ---
(11, 'ttf', 'Type0', 'CIDFont+F1', 'F1', 'Identity-H', 0)
(19, 'ttf', 'Type0', 'CIDFont+F2', 'F2', 'Identity-H', 0)
(27, 'ttf', 'Type0', 'CIDFont+F3', 'F3', 'Identity-H', 0)


In [18]:
import pymupdf

page = doc[38]

print("NORMAL EXTRACTION:")
print(page.get_text()[:600])

print("\nRAW DICT EXTRACTION:")
raw = page.get_text("dict")

for block in raw["blocks"]:
    if "lines" in block:
        for line in block["lines"]:
            for span in line["spans"]:
                print(span["text"])
                

NORMAL EXTRACTION:
31.01.2024 
 
37 
 
माननीय अÅय±: आइटम नं. 1, महासिचव । 
12.58 hrs 
PRESIDENT’S ADDRESS 
 
SECRETARY-GENERAL: I beg to lay on the Table a copy of the President’s 
Address to both Houses of Parliament assembled together on the 31st of January, 
2024. 
माननीय सदÖयगण, 
इस नए संसद भवन म¤ यह मेरा पहला संबोधन है। आज़ादी के अमृतकाल कì शुŁआत म¤ यह 
भÓय भवन बना है। यहां एक भारत ®ेķ भारत कì महक भी है।भारत कì सËयता और संÖकृित कì 
चेतना भी है। इसम¤, हमारी लोकतांिýक और संसदीय परंपराओं के सÌमान का ÿण भी है।साथ ही, 
21वé सदी के नए भारत के िलए, नई परंपराओं के िनमाªण का संकÐप भी है।मुझे पूरा िवĵास है िक इस 
न

RAW DICT EXTRACTION:
31.01.2024 
 
37 
 
माननीय अ
Å
य
±
:
 आइटम नं. 1, महासिचव ।
 
12.58 hrs 
PRESIDENT’S ADDRESS

 
 
SECRETARY-GENERAL: 
I beg to lay on the Table a copy of the President’s 
Address to both Houses of Parliament assembled together on the 31
st
 of January, 
2024.
 

माननीय सद
Ö
यगण, 
इस नए संसद भवन म
¤
 यह मेरा पहला संबोधन है। आज़ादी के अमृतकाल क
ì
 शु
Ł
आत 

In [19]:
for font in doc[38].get_fonts(full=True):
    xref = font[0]
    print(f"\nFont: {font[3]} | xref: {xref}")

    try:
        cmap = doc.extract_font(xref)
        print("Font extraction metadata:", cmap[:3])
    except Exception as e:
        print("Could not extract font:", e)


Font: CIDFont+F1 | xref: 11
Font extraction metadata: ('CIDFont+F1', 'ttf', 'Type0')

Font: CIDFont+F2 | xref: 19
Font extraction metadata: ('CIDFont+F2', 'ttf', 'Type0')

Font: CIDFont+F7 | xref: 174
Font extraction metadata: ('CIDFont+F7', 'ttf', 'Type0')

Font: CIDFont+F8 | xref: 182
Font extraction metadata: ('CIDFont+F8', 'ttf', 'Type0')


In [20]:
%pip install pdfplumber

Note: you may need to restart the kernel to use updated packages.


In [21]:
import pdfplumber

with pdfplumber.open(pdf_path) as pdf:
    print("PDF pages:", len(pdf.pages))
    print(pdf.pages[38].extract_text()[:1500])

PDF pages: 89
31.01.2024 37
माननीय अ(cid:197)य(cid:177): आइटम न.ं 1, महासिचव ।
12.58 hrs

PRESIDENT’S ADDRESS
SECRETARY-GENERAL: I beg to lay on the Table a copy of the President’s
Address to both Houses of Parliament assembled together on the 31st of January,
2024.

माननीय सद(cid:214)यगण,
इस नए संसद भवन म(cid:164) यह मेरा पहला संबोधन है। आज़ादी के अमृतकाल क(cid:236) शु(cid:321)आत म (cid:164) यह
भ(cid:211)य भवन बना है। यहां एक भारत (cid:174)े(cid:311) भारत क(cid:236) महक भी है।भारत क(cid:236) स(cid:203)यता और सं(cid:214)कृित क(cid:236)
चेतना भी है। इसम(cid:164), हमारी लोकतांि(cid:253)क और संसदीय परपं राओ ं के स(cid:204)मान का (cid:255)ण भी है।साथ ही,
21व(cid:233) सदी के नए भारत के िलए, नई परपं राओ ंके िनमा(cid:170)ण का संक(cid:208)प भी है।मझु े पूरा िव(cid:309)ास है िक इस
नए भवन म(cid:164) नीितय(cid:332) पर साथ(cid:170)क संवाद होगा। ऐसी नीितयां जो आज़ादी के अमृतकाल म(cid:164) िवकिसत
भारत का िनमा(cid:170)ण करग(cid:164) ी। म(cid:167) आप सभी को अपनी शुभकामनाए ं देती ह(cid:243)ं।
माननीय स

In [22]:
import subprocess

result = subprocess.run(
    [r"C:\Program Files\Tesseract-OCR\tesseract.exe", "--list-langs"],
    capture_output=True,
    text=True
)

print(result.stdout)

List of available languages in "C:\Program Files\Tesseract-OCR/tessdata/" (124):
afr
amh
ara
asm
aze
aze_cyrl
bel
ben
bod
bos
bre
bul
cat
ceb
ces
chi_sim
chi_sim_vert
chi_tra
chi_tra_vert
chr
cos
cym
dan
deu
deu_latf
div
dzo
ell
eng
enm
epo
equ
est
eus
fao
fas
fil
fin
fra
frm
fry
gla
gle
glg
grc
guj
hat
heb
hin
hrv
hun
hye
iku
ind
isl
ita
ita_old
jav
jpn
jpn_vert
kan
kat
kat_old
kaz
khm
kir
kmr
kor
lao
lat
lav
lit
ltz
mal
mar
mkd
mlt
mon
mri
msa
mya
nep
nld
nor
oci
ori
osd
pan
pol
por
pus
que
ron
rus
san
sin
slk
slv
snd
spa
spa_old
sqi
srp
srp_latn
sun
swa
swe
syr
tam
tat
tel
tgk
tha
tir
ton
tur
uig
ukr
urd
uzb
uzb_cyrl
vie
yid
yor



In [23]:
import shutil

print(shutil.which("tesseract"))

None


In [24]:
import os
import shutil
import pymupdf

tesseract_dir = r"C:\Program Files\Tesseract-OCR"
tesseract_exe = os.path.join(tesseract_dir, "tesseract.exe")
tessdata_dir = os.path.join(tesseract_dir, "tessdata")

print("Executable exists:", os.path.exists(tesseract_exe))
print("Hindi data exists:", os.path.exists(os.path.join(tessdata_dir, "hin.traineddata")))

os.environ["PATH"] = tesseract_dir + os.pathsep + os.environ["PATH"]
pymupdf.TESSDATA_PREFIX = tessdata_dir

print("Tesseract found:", shutil.which("tesseract"))

Executable exists: True
Hindi data exists: True
Tesseract found: C:\Program Files\Tesseract-OCR\tesseract.EXE


In [25]:
import pymupdf

pymupdf.TESSDATA_PREFIX = r"C:\Program Files\Tesseract-OCR\tessdata"

doc = pymupdf.open(pdf_path)
page = doc[38]  # PDF page 39: Hindi address

textpage = page.get_textpage_ocr(
    language="hin+eng",
    dpi=300,
    full=True
)

ocr_text = page.get_text(textpage=textpage)
print(ocr_text[:2000])

34.04.2024
37
माननीयअध्यक्ष: आइटमनं.
, महासचिव
|
42.58 hrs
PRESIDENT’S ADDRESS*
SECRETARY-GENERAL:
| beg to lay on the Table a copy of the President’s
Address to both Houses of Parliament assembled together on the 3* of January,
2024.
“HAART सदस्यगण,
इसनएसंसदभवनमेंयहमेरापहलासंबोधनहै।आज़ादीकेअमृतकालकीशुरुआतमेंयह
भव्यभवनबनाहै।यहांएकभारतश्रेष्ठभारतकीमहकभीहै।भारतकीसभ्यताऔरसंस्कृतिकी
चेतनाभीहै।इसमें, हमारीलोकतांत्रिकऔरसंसदीयपरंपराओंकेसम्मानकाप्रणभीहै।साथही,
24वींसदीकेनएभारतकेलिए, नईपरंपराओंकेनिर्माणकासंकल्पभीहै।मुझेपूराविश्वासहै
किइस
नएभवनमेंनीतियोंपरसार्थकसंवादहोगा।ऐसीनीतियां
जोआज़ादीकेअमृतकालमेंविकसित
भारतकानिर्माणकरेंगी।मैं
आपसभीकोअपनीशुभकामनाएंदेतीहूं।
माननीयसदस्यगण, यहहमारेसंविधानकेलागूहोनेकाभी75वां
वर्षहै।इसीकालखंडमें
आज़ादीके75 वर्षकाउत्सव, अमृतमहोत्सवभीसंपननहुआहै
।इसदौरानदेशभरमेंअनेक
कार्यक्रमहुए
| देशनेअपनेगुमनामस्वतंत्रतासेनानियों
कोयादकिया
| 75 सालबादयुवापीढ़ीने
फिरस्वतन्त्रतासंग्रामकेउसकालखंडकोजिया|
इसउत्सवकेदौरानमेरीमाटी, मेरादेशअभियानकेतहत, देशभरकेहरगाँवकीमिट्टीके
साथअमृतकलशदि

In [26]:
import pymupdf

doc = pymupdf.open(pdf_path)

# PDF page 60: English version of the address
page = doc[59]

direct_text = page.get_text()

ocr_page = page.get_textpage_ocr(
    language="eng",
    dpi=300,
    full=True
)
ocr_text = page.get_text(textpage=ocr_page)

print("=== DIRECT EXTRACTION ===")
print(direct_text[:1000])

print("\n=== OCR EXTRACTION ===")
print(ocr_text[:1000])

=== DIRECT EXTRACTION ===
31.01.2024 
 
58 
 
Honorable Members, 
This is my first address in this new Parliament building. This magnificent 
building has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is 
imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to 
India’s civilization and culture. It also resonates with the resolve to respect our 
democratic and parliamentary traditions. Moreover, it embodies the commitment 
to forge new traditions for the new India of the 21st century. I am confident that 
this new building will be witness to productive dialogue on policies -- policies that 
will shape the development of ‘Viksit Bharat’ in the Amrit Kaal of our 
independence. I extend my best wishes to all of you. 
Honorable Members, this year is also the 75th year of adoption of our 
Constitution. During this period, Amrit Mahotsav, the celebration of 75 years 
of independence, was completed. During this period, many programmes were 
organized ac

In [27]:
import pymupdf

doc = pymupdf.open(pdf_path)

for i, page in enumerate(doc):
    direct = page.get_text().strip()
    print(
        f"PDF page {i + 1:02}: "
        f"{len(direct):5} characters | "
        f"{direct[:100].replace(chr(10), ' ')}"
    )

PDF page 01:   325 characters | Seventeenth Series, Vol. XXIX  No. 1                                      Wednesday, January 31, 202
PDF page 02:   792 characters | EDITORIAL BOARD      Utpal Kumar Singh  Secretary-General  Lok Sabha    Mamta Kemwal   Joint Secreta
PDF page 03:   701 characters | 31.01.2024    1    C O N T E N T S    Seventeenth Series, Vol. XXIX, Fifteenth Session, 2024/1945 (S
PDF page 04:   875 characters | 31.01.2024    2    ALPHABETICAL LIST OF MEMBERS OF SEVENTEENTH LOK SABHA  Abdullah, Dr. Farooq (Srin
PDF page 05:   867 characters | 31.01.2024    3    Bachegowda, Shri B.N. (Chikkballapur)  Badal, Shri Sukhbir Singh (Firozpur)  Bada
PDF page 06:   863 characters | 31.01.2024    4    Bhamre, Dr. Subhash Ramrao (Dhule)  Bharat, Shri Margani (Rajahmundry)  Bhargava,
PDF page 07:   907 characters | 31.01.2024    5    Chatterjee, Shrimati Locket (Hooghly)  Chaudhary, Shri P. P. (Pali)  Chaudhary, S
PDF page 08:   845 characters | 31.01.2024    6    Darbar, Shri Chatt

In [28]:
import pymupdf

pymupdf.TESSDATA_PREFIX = r"C:\Program Files\Tesseract-OCR\tessdata"

doc = pymupdf.open(pdf_path)

page_number = 48  # PDF page 49, in the Hindi address
page = doc[page_number - 1]

textpage = page.get_textpage_ocr(
    language="hin+eng",
    dpi=300,
    full=True
)

sample_text = page.get_text(textpage=textpage)
print(sample_text[:2000])

34.04.2024
46
सेदेशकीलाखोंमहिलाओंकोबहुतमददमिलीहै।मेरीसरकारनेमहिलाओंकोपहलीबारसशस्त्र
बलों
मेंपरमानेंटकमीशनदियाहै।पहलीबारसैनिकस्कूलोंऔरराष्ट्रीयरक्षाअकादमीमेंमहिला
कैडेट्सकोप्रवेशदियागयाहै।आजमहिलाएंफाइटरपायलटभीहैंऔरनौसेनाकेजहाजकोभी
पहलीबारPAS करYel Sl मुद्रायोजनाकेतहतजो46 करोड़सेज्यादालोनदिएगए
हैंउनमें
करीब34 करोड़सेज्यादालोनमहिलाओंकोमिले
हैं।इसयोजनासेलाभलेकरकरोड़ोंमहिलाओं
नेस्वरोजगारशुरुकियाहै।
माननीयसदस्यगण, मेरीसरकारआजखेतीकोअधिकलाभकारीबनानेपरबलदे
रहीहै।
हमारायहप्रयासहैकिखेतीमेंलागतकमहोऔरलाभअधिकहो
| मेरीसरकारनेपहलीबार
0
करोड़सेअधिकछोटेकिसानोंकोभीदेशकीकृषिनीतिऔरयोजनाओं
मेंप्रमुखतादीहै।पीएम-
किसानसम्माननिधिकेतहतअबतक2 लाख80 हज़ारकरोड़रुपए, किसानोंकोमिलचुके
हैं।
40
सालों
मेंकिसानोंकेलिएबैंकसेआसानलोनमें3 गुनावृद्धिकीगईहै।प्रधानमंत्रीफसलबीमा
योजनाकेतहतकिसानों
ने30 हज़ारकरोड़रुपएप्रीमियमभरा।इसकेबदले
उन्हें
डेढ़लाखकरोड़
रुपएकाFors मिलाSl पिछले
40
वर्षोंमें, लगभग
48 लाखकरोड़रुपएMSP केरूपमेंधानऔर
गेहूं
कीखेतीकरनेवालेकिसानोंकोमिले
हैं।यह2044 सेपहलेके
40 सालोंकीतुलनामेंढाईगुना
अधिकहै।पहलेतिलहनऔरदल

In [29]:
import pymupdf
from pathlib import Path

pdf_path = Path("data/raw/Floor.pdf")
doc = pymupdf.open(pdf_path)

records = []

for page_index in range(59, 86):  # PDF pages 60–86
    page = doc[page_index]
    text = page.get_text("text").strip()

    if text:
        records.append({
            "document_id": "president_address_2024_01_31",
            "source_file": pdf_path.name,
            "pdf_page": page_index + 1,
            "language": "en",
            "extraction_method": "direct",
            "text": text
        })

print("Pages extracted:", len(records))
print("First record:")
print(records[0])

Pages extracted: 27
First record:
{'document_id': 'president_address_2024_01_31', 'source_file': 'Floor.pdf', 'pdf_page': 60, 'language': 'en', 'extraction_method': 'direct', 'text': '31.01.2024 \n \n58 \n \nHonorable Members, \nThis is my first address in this new Parliament building. This magnificent \nbuilding has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is \nimbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to \nIndia’s civilization and culture. It also resonates with the resolve to respect our \ndemocratic and parliamentary traditions. Moreover, it embodies the commitment \nto forge new traditions for the new India of the 21st century. I am confident that \nthis new building will be witness to productive dialogue on policies -- policies that \nwill shape the development of ‘Viksit Bharat’ in the Amrit Kaal of our \nindependence. I extend my best wishes to all of you. \nHonorable Members, this year is also the 75th year of adoption o

In [30]:
import json
from pathlib import Path

output_path = Path("data/processed/president_address_en.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved to:", output_path.resolve())
print("Records saved:", len(records))

Saved to: C:\Users\Tony\Documents\Lok_Sabha_AI\data\processed\president_address_en.jsonl
Records saved: 27


In [31]:
with output_path.open("r", encoding="utf-8") as f:
    first_record = json.loads(f.readline())

print("PDF page:", first_record["pdf_page"])
print("Language:", first_record["language"])
print("Extraction:", first_record["extraction_method"])
print(first_record["text"][:500])

PDF page: 60
Language: en
Extraction: direct
31.01.2024 
 
58 
 
Honorable Members, 
This is my first address in this new Parliament building. This magnificent 
building has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is 
imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to 
India’s civilization and culture. It also resonates with the resolve to respect our 
democratic and parliamentary traditions. Moreover, it embodies the commitment 
to forge new traditions for the new India of the 21st centu


In [32]:
import re
def normalize_text(text):
    text = text.strip()
    text = re.sub(r"[ \t]+"," ",text)
    text = re.sub(r"\n{3,}","\n\n",text)
    return text

In [33]:
original_text = first_record["text"]
cleaned_text = normalize_text(original_text)

print("BEFORE:\n")
print(original_text[:700])

print("\nAFTER:\n")
print(cleaned_text[:700])

BEFORE:

31.01.2024 
 
58 
 
Honorable Members, 
This is my first address in this new Parliament building. This magnificent 
building has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is 
imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to 
India’s civilization and culture. It also resonates with the resolve to respect our 
democratic and parliamentary traditions. Moreover, it embodies the commitment 
to forge new traditions for the new India of the 21st century. I am confident that 
this new building will be witness to productive dialogue on policies -- policies that 
will shape the development of ‘Viksit Bharat’ in the Amrit Kaal of our 
independence. I 

AFTER:

31.01.2024 
 
58 
 
Honorable Members, 
This is my first address in this new Parliament building. This magnificent 
building has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is 
imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to 
Indi

In [34]:
for record in records:
    record["cleaned_text"] = normalize_text(record["text"])

print("Records processed:", len(records))
print("Original text preserved:", "text" in records[0])
print("Cleaned text available:", "cleaned_text" in records[0])

Records processed: 27
Original text preserved: True
Cleaned text available: True


In [35]:
changed_pages = [
    record for record in records
    if record["text"] != record["cleaned_text"]
]

print("Pages changed by normalization:", len(changed_pages))

for record in changed_pages[:3]:
    print("\nPDF page:", record["pdf_page"])
    print("BEFORE:", repr(record["text"][:200]))
    print("AFTER: ", repr(record["cleaned_text"][:200]))

Pages changed by normalization: 20

PDF page: 60
BEFORE: '31.01.2024 \n \n58 \n \nHonorable Members, \nThis is my first address in this new Parliament building. This magnificent \nbuilding has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is \nimbu'
AFTER:  '31.01.2024 \n \n58 \n \nHonorable Members, \nThis is my first address in this new Parliament building. This magnificent \nbuilding has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is \nimbu'

PDF page: 61
BEFORE: '31.01.2024 \n \n59 \n \nIt was during the Amrit Mahotsav that, A statue of Netaji Subhash Chandra \nBose was installed on “Kartavya Path”. A museum dedicated to all the Prime \nMinisters of the country was '
AFTER:  '31.01.2024 \n \n59 \n \nIt was during the Amrit Mahotsav that, A statue of Netaji Subhash Chandra \nBose was installed on “Kartavya Path”. A museum dedicated to all the Prime \nMinisters of the country was '

PDF page: 63
BEFORE: '31.01.2024 \n \n61 \n \nthe national interes

In [36]:
for record in changed_pages[:3]:
    print(f"\nPDF page: {record['pdf_page']}")

    print("BEFORE:")
    print(repr(record["text"][:150]))

    print("AFTER:")
    print(repr(record["cleaned_text"][:150]))


PDF page: 60
BEFORE:
'31.01.2024 \n \n58 \n \nHonorable Members, \nThis is my first address in this new Parliament building. This magnificent \nbuilding has been constructed at t'
AFTER:
'31.01.2024 \n \n58 \n \nHonorable Members, \nThis is my first address in this new Parliament building. This magnificent \nbuilding has been constructed at t'

PDF page: 61
BEFORE:
'31.01.2024 \n \n59 \n \nIt was during the Amrit Mahotsav that, A statue of Netaji Subhash Chandra \nBose was installed on “Kartavya Path”. A museum dedicat'
AFTER:
'31.01.2024 \n \n59 \n \nIt was during the Amrit Mahotsav that, A statue of Netaji Subhash Chandra \nBose was installed on “Kartavya Path”. A museum dedicat'

PDF page: 63
BEFORE:
'31.01.2024 \n \n61 \n \nthe national interest, for which the people of the country were waiting for decades. \nFor centuries, there was an aspiration to co'
AFTER:
'31.01.2024 \n \n61 \n \nthe national interest, for which the people of the country were waiting for decades. \nFor centu

In [37]:
for record in changed_pages[:3]:
    before = record["text"]
    after = record["cleaned_text"]

    print(
        f"Page {record['pdf_page']}: "
        f"{len(before)} -> {len(after)} characters; "
        f"{len(before) - len(after)} removed"
    )

Page 60: 1595 -> 1593 characters; 2 removed
Page 61: 1669 -> 1667 characters; 2 removed
Page 63: 1335 -> 1317 characters; 18 removed


In [38]:
for record in changed_pages[:3]:
    before = record["text"]
    after = record["cleaned_text"]

    print(f"\nPDF page: {record['pdf_page']}")

    # Show positions where the text differs
    for i, (old, new) in enumerate(zip(before, after)):
        if old != new:
            start = max(0, i - 40)
            end = min(len(before), i + 60)

            print("BEFORE:", repr(before[start:end]))
            print("AFTER: ", repr(after[start:end]))
            break


PDF page: 60
BEFORE: 'n. Over \n2 lakh plaques were installed.  More than three crore people took the oath of the \n‘Panch P'
AFTER:  'n. Over \n2 lakh plaques were installed. More than three crore people took the oath of the \n‘Panch Pr'

PDF page: 61
BEFORE: 'declared in the memory of “Sahibzaade”.  Birth anniversary of \nBhagwan Birsa Munda was declared as “'
AFTER:  'declared in the memory of “Sahibzaade”. Birth anniversary of \nBhagwan Birsa Munda was declared as “J'

PDF page: 63
BEFORE: 'e had said, \nिमशु मोर देह ए देश मािटरे,    \nदेशबासी चािल जाआÆतु िपिठरे।  \nदेशर Öवराºय-पथे जेते गाड़, '
AFTER:  'e had said, \nिमशु मोर देह ए देश मािटरे, \nदेशबासी चािल जाआÆतु िपिठरे। \nदेशर Öवराºय-पथे जेते गाड़, \nपूŁ'


In [39]:
import json
from pathlib import Path

output_path = Path("data/processed/president_address_en_cleaned.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    for record in records:
        cleaned_record = {
            **record,
            "text": record["cleaned_text"]
        }
        cleaned_record.pop("cleaned_text", None)

        f.write(json.dumps(cleaned_record, ensure_ascii=False) + "\n")

print("Saved cleaned records:", len(records))
print("Output file:", output_path)

Saved cleaned records: 27
Output file: data\processed\president_address_en_cleaned.jsonl


In [40]:
with output_path.open("r", encoding="utf-8") as f:
    saved_records = [json.loads(line) for line in f if line.strip()]

print("Records loaded:", len(saved_records))
print("First record page:", saved_records[0]["pdf_page"])
print("Last record page:", saved_records[-1]["pdf_page"])
print("Text field exists:", "text" in saved_records[0])
print("Temporary field removed:", "cleaned_text" not in saved_records[0])

Records loaded: 27
First record page: 60
Last record page: 86
Text field exists: True
Temporary field removed: True


In [41]:
page_text = saved_records[0]["text"]
print(page_text)

31.01.2024 
 
58 
 
Honorable Members, 
This is my first address in this new Parliament building. This magnificent 
building has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is 
imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to 
India’s civilization and culture. It also resonates with the resolve to respect our 
democratic and parliamentary traditions. Moreover, it embodies the commitment 
to forge new traditions for the new India of the 21st century. I am confident that 
this new building will be witness to productive dialogue on policies -- policies that 
will shape the development of ‘Viksit Bharat’ in the Amrit Kaal of our 
independence. I extend my best wishes to all of you. 
Honorable Members, this year is also the 75th year of adoption of our 
Constitution. During this period, Amrit Mahotsav, the celebration of 75 years 
of independence, was completed. During this period, many programmes were 
organized across the country. The coun

In [42]:
sentences = re.split(r"(\.\s+)", page_text)
for sentence in sentences:
    print(sentence)

31.01.2024 
 
58 
 
Honorable Members, 
This is my first address in this new Parliament building
. 
This magnificent 
building has been constructed at the beginning of “Azadi ka Amrit Kaal”
. 
It is 
imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to 
India’s civilization and culture
. 
It also resonates with the resolve to respect our 
democratic and parliamentary traditions
. 
Moreover, it embodies the commitment 
to forge new traditions for the new India of the 21st century
. 
I am confident that 
this new building will be witness to productive dialogue on policies -- policies that 
will shape the development of ‘Viksit Bharat’ in the Amrit Kaal of our 
independence
. 
I extend my best wishes to all of you
. 

Honorable Members, this year is also the 75th year of adoption of our 
Constitution
. 
During this period, Amrit Mahotsav, the celebration of 75 years 
of independence, was completed
. 
During this period, many programmes were 
organized across the

In [43]:
print("Number of pieces:", len(sentences))

for sentence in sentences[:5]:
    print(repr(sentence))

Number of pieces: 37
'31.01.2024 \n \n58 \n \nHonorable Members, \nThis is my first address in this new Parliament building'
'. '
'This magnificent \nbuilding has been constructed at the beginning of “Azadi ka Amrit Kaal”'
'. '
'It is \nimbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to \nIndia’s civilization and culture'


In [44]:
cleaned_sentences = []
for sentence in sentences:
    s = sentence.replace("\n"," ")
    s = normalize_text(s)
    cleaned_sentences.append(s)
for sentence in cleaned_sentences:
    print(sentence)

31.01.2024 58 Honorable Members, This is my first address in this new Parliament building
.
This magnificent building has been constructed at the beginning of “Azadi ka Amrit Kaal”
.
It is imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to India’s civilization and culture
.
It also resonates with the resolve to respect our democratic and parliamentary traditions
.
Moreover, it embodies the commitment to forge new traditions for the new India of the 21st century
.
I am confident that this new building will be witness to productive dialogue on policies -- policies that will shape the development of ‘Viksit Bharat’ in the Amrit Kaal of our independence
.
I extend my best wishes to all of you
.
Honorable Members, this year is also the 75th year of adoption of our Constitution
.
During this period, Amrit Mahotsav, the celebration of 75 years of independence, was completed
.
During this period, many programmes were organized across the country
.
The country remem

In [45]:
print(len(cleaned_sentences))

37


In [46]:
chunks = []
current_chunk = ""

for sentence in cleaned_sentences:
    if len(current_chunk + " " + sentence) > 500:
        chunks.append(current_chunk)
        current_chunk = sentence
    else:
        current_chunk += " " + sentence


if current_chunk:
    chunks.append(current_chunk)

In [47]:
print("Number of chunks:", len(chunks))
print("Chunk lengths:", [len(chunk) for chunk in chunks])

Number of chunks: 4
Chunk lengths: [497, 485, 496, 104]


In [48]:
for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk[:150])


--- Chunk 1 ---
 31.01.2024 58 Honorable Members, This is my first address in this new Parliament building . This magnificent building has been constructed at the beg

--- Chunk 2 ---
I am confident that this new building will be witness to productive dialogue on policies -- policies that will shape the development of ‘Viksit Bharat

--- Chunk 3 ---
The country remembered its unsung freedom fighters . After 75 years, the young generation relived that period of freedom struggle . During this campai

--- Chunk 4 ---
Over two crore trees were planted . More than 16 crore people uploaded their selfies with the Tricolour.


In [49]:
test_text = "building . This magnificent building"
print(re.sub(r"\s+([.!?])", r"\1", test_text))

building. This magnificent building


In [50]:
chunks = [
    re.sub(r"\s+([.!?])", r"\1", chunk)
    for chunk in chunks
]

print([len(chunk) for chunk in chunks])
print(chunks[0][:150])

[492, 480, 489, 103]
 31.01.2024 58 Honorable Members, This is my first address in this new Parliament building. This magnificent building has been constructed at the begi


In [51]:
print(saved_records[0].keys())
print(saved_records[0]["pdf_page"])
print(saved_records[0]["text"][:200])

dict_keys(['document_id', 'source_file', 'pdf_page', 'language', 'extraction_method', 'text'])
60
31.01.2024 
 
58 
 
Honorable Members, 
This is my first address in this new Parliament building. This magnificent 
building has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is 
imbu


In [52]:
parts = re.split(r"\n\s*\n", saved_records[0]["text"].strip())

for i, part in enumerate(parts):
    print(i, repr(part[:100]))

0 '31.01.2024 '
1 '58 '
2 'Honorable Members, \nThis is my first address in this new Parliament building. This magnificent \nbuil'


In [53]:
speech_text = parts[2]
print(speech_text[:150])

Honorable Members, 
This is my first address in this new Parliament building. This magnificent 
building has been constructed at the beginning of “Aza


In [54]:
sentences = re.split(r"(?<=\.)\s+", speech_text)
for sentence in sentences[:3]:
    print(repr(sentence))

'Honorable Members, \nThis is my first address in this new Parliament building.'
'This magnificent \nbuilding has been constructed at the beginning of “Azadi ka Amrit Kaal”.'
'It is \nimbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to \nIndia’s civilization and culture.'


In [55]:
cleaned_sentences = []
for sentence in sentences:
    cleaned_sentences.append(sentence.replace("\n"," "))
print(cleaned_sentences)

['Honorable Members,  This is my first address in this new Parliament building.', 'This magnificent  building has been constructed at the beginning of “Azadi ka Amrit Kaal”.', 'It is  imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to  India’s civilization and culture.', 'It also resonates with the resolve to respect our  democratic and parliamentary traditions.', 'Moreover, it embodies the commitment  to forge new traditions for the new India of the 21st century.', 'I am confident that  this new building will be witness to productive dialogue on policies -- policies that  will shape the development of ‘Viksit Bharat’ in the Amrit Kaal of our  independence.', 'I extend my best wishes to all of you.', 'Honorable Members, this year is also the 75th year of adoption of our  Constitution.', 'During this period, Amrit Mahotsav, the celebration of 75 years  of independence, was completed.', 'During this period, many programmes were  organized across the country.'

In [56]:
cleaned_sentences = [normalize_text(s.replace("\n", " ")) for s in sentences]

In [58]:
%pip install -U langchain-text-splitters langchain-community langchain-huggingface sentence-transformers faiss-cpu


  Attempting uninstall: langchain-huggingface
    Found existing installation: langchain-huggingface 1.2.2
    Uninstalling langchain-huggingface-1.2.2:
      Successfully uninstalled langchain-huggingface-1.2.2
Note: you may need to restart the kernel to use updated packages.


In [59]:
import langchain_text_splitters
import langchain_community
import langchain_huggingface
import faiss

print("RAG dependencies imported successfully.")

C:\Users\Tony\AppData\Local\Temp\ipykernel_18120\128000402.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


RAG dependencies imported successfully.


In [60]:
import json
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

input_path = Path("data/processed/president_address_en_cleaned.jsonl")

records = []

with input_path.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("Records loaded:", len(records))
print("First record keys:", records[0].keys())

Records loaded: 27
First record keys: dict_keys(['document_id', 'source_file', 'pdf_page', 'language', 'extraction_method', 'text'])


In [61]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [63]:
from langchain_core.documents import Document
documents = []
for record in records:
    document = Document(
         page_content=record["text"],
        metadata={
            "document_id": record["document_id"],
            "source_file": record["source_file"],
            "pdf_page": record["pdf_page"],
            "language": record["language"],
            "extraction_method": record["extraction_method"]
        }
    )
    documents.append(document)
    
chunks = text_splitter.split_documents(documents)

print("Pages loaded:", len(documents))
print("Chunks created:", len(chunks))
print("\nFirst chunk metadata:", chunks[0].metadata)
print("\nFirst chunk preview:\n", chunks[0].page_content[:300])

Pages loaded: 27
Chunks created: 103

First chunk metadata: {'document_id': 'president_address_2024_01_31', 'source_file': 'Floor.pdf', 'pdf_page': 60, 'language': 'en', 'extraction_method': 'direct'}

First chunk preview:
 31.01.2024 
 
58 
 
Honorable Members, 
This is my first address in this new Parliament building. This magnificent 
building has been constructed at the beginning of “Azadi ka Amrit Kaal”. It is 
imbued with the fragrance of ‘Ek Bharat Shreshtha Bharat’ and is a testimony to 
India’s civilization an


In [64]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [67]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS index created successfully.")
print("Chunks indexed:", vector_store.index.ntotal)

FAISS index created successfully.
Chunks indexed: 103


In [68]:
question = "What did the President say about India's development?"

retrieved_docs = vector_store.similarity_search(
    question,
    k=3
)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Source {i} ---")
    print("PDF page:", doc.metadata["pdf_page"])
    print(doc.page_content[:500])


--- Source 1 ---
PDF page: 77
taken keeping this goal in mind. 
Honorable Members, today every agency in the world is assured of India’s 
rapid development. The assessments of the national and international agencies 
are based on India’s policies. Record investments in infrastructure and policy 
reforms are further boosting investors’ confidence. The preference of Indians for a 
stable and strong government with full majority has also renewed the confidence

--- Source 2 ---
PDF page: 80
this. 
 
My government is engaged in making India a major space power in the 
world. It is a means to improve human life. Besides, this also attempts to increase 
India’s share in the space economy. Major decisions have been taken to expand 
India’s space programme. This has led to the formation of many new space 
startups. The day is not far when India’s Gaganyaan will reach space. 
Honorable Members, my government has made India one of the world’s

--- Source 3 ---
PDF page: 83
held in India. The un

In [69]:
%pip install -U langchain-ollama


   ---------------------------------------- 2/2 [langchain-ollama]

Note: you may need to restart the kernel to use updated packages.


In [74]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

response = llm.invoke(
    "Explain the purpose of the Indian Parliament in two sentences."
)

print(response.content)

The Indian Parliament is the supreme legislative body of India, comprising the Lok Sabha (House of the People) and the Rajya Sabha (Council of States), which together exercise the legislative powers of the country. The primary purpose of the Indian Parliament is to make, amend, and repeal laws, as well as to exercise oversight over the executive branch and ensure accountability to the people, while also representing the interests of various states and regions in the country.


In [75]:
from langchain_core.prompts import ChatPromptTemplate

question = "What did the President say about India's development?"

# 1. Retrieve relevant passages
retrieved_docs = vector_store.similarity_search(question, k=3)

# 2. Attach source-page labels to the evidence
context = "\n\n".join(
    f"[PDF p. {doc.metadata['pdf_page']}]\n{doc.page_content}"
    for doc in retrieved_docs
)

# 3. Instruct Llama to answer from the evidence
prompt = ChatPromptTemplate.from_template("""
You are a parliamentary research assistant.

Rules:
- Answer using only the supplied source passages.
- Do not add unsupported facts or assumptions.
- Cite supporting pages using the exact format [PDF p. N].
- Include citations next to the claims they support.
- If the evidence is insufficient, say so clearly.

SOURCE PASSAGES:
{context}

QUESTION:
{question}

ANSWER:
""")

messages = prompt.format_messages(
    context=context,
    question=question
)

response = llm.invoke(messages)

print(response.content)

print("\nRetrieved source pages:")
print(sorted({doc.metadata["pdf_page"] for doc in retrieved_docs}))

The President stated that "every agency in the world is assured of India's rapid development" [PDF p. 77].

Retrieved source pages:
[77, 80, 83]


In [72]:
for doc in retrieved_docs:
    print(f"PDF page: {doc.metadata['pdf_page']}")
    print(doc.page_content[:400])
    print("-" * 50)

PDF page: 77
taken keeping this goal in mind. 
Honorable Members, today every agency in the world is assured of India’s 
rapid development. The assessments of the national and international agencies 
are based on India’s policies. Record investments in infrastructure and policy 
reforms are further boosting investors’ confidence. The preference of Indians for a 
stable and strong government with full majority 
--------------------------------------------------
PDF page: 80
this. 
 
My government is engaged in making India a major space power in the 
world. It is a means to improve human life. Besides, this also attempts to increase 
India’s share in the space economy. Major decisions have been taken to expand 
India’s space programme. This has led to the formation of many new space 
startups. The day is not far when India’s Gaganyaan will reach space. 
Honorable Mem
--------------------------------------------------
PDF page: 83
held in India. The unanimous adoption of the Delhi Declar

In [91]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
You are a parliamentary research assistant.

Answer the question using only the supplied source passages.
Do not add facts that are not supported by the passages.
If the evidence is insufficient, say so.
Do not write page numbers or citations in your answer.
The application will add source references separately.

SOURCE PASSAGES:
{context}

QUESTION:
{question}

ANSWER:
""")

def ask_parliament(question, k=3):
    docs = vector_store.similarity_search(question, k=k)

    context = "\n\n".join(
        f"[Retrieved PDF page {doc.metadata['pdf_page']}]\n"
        f"{doc.page_content}"
        for doc in docs
    )

    response = llm.invoke(
        rag_prompt.format_messages(
            context=context,
            question=question
        )
    )

    answer_text = response.content.strip()

    # Generate citations from actual retrieved metadata
    source_pages = sorted({
        int(doc.metadata["pdf_page"])
        for doc in docs
    })

    print("Question:", question)
    print("\nAnswer:\n", answer_text)
    print(
        "\nSources consulted:",
        ", ".join(f"[PDF p. {page}]" for page in source_pages)
    )

    return answer_text, docs

In [92]:
answer, sources = ask_parliament(
    "What did the President say about India's space programme?"
)

Question: What did the President say about India's space programme?

Answer:
 The President mentioned that major decisions have been taken to expand India's space programme, which has led to the formation of many new space startups.

Sources consulted: [PDF p. 66], [PDF p. 80], [PDF p. 84]


In [84]:
import re

def check_citations(answer, docs):
    retrieved_pages = {
        int(doc.metadata["pdf_page"])
        for doc in docs
    }

    cited_pages = {
        int(page)
        for page in re.findall(
            r"\[PDF p\.\s*(\d+)\]",
            answer
        )
    }

    invalid_pages = cited_pages - retrieved_pages

    print("Retrieved pages:", sorted(retrieved_pages))
    print("Cited pages:", sorted(cited_pages))
    print("Invalid citations:", sorted(invalid_pages))

    return len(invalid_pages) == 0

In [85]:
citation_valid = check_citations(answer, sources)
print("Citation validation passed:", citation_valid)

Retrieved pages: [66, 80, 84]
Cited pages: [64, 80]
Invalid citations: [64]
Citation validation passed: False


In [86]:
import re

def validate_rag_answer(answer, docs):
    retrieved_pages = {
        int(doc.metadata["pdf_page"])
        for doc in docs
    }

    cited_pages = {
        int(page)
        for page in re.findall(
            r"\[PDF p\.\s*(\d+)\]",
            answer
        )
    }

    invalid_pages = cited_pages - retrieved_pages

    if invalid_pages:
        print(
            "WARNING: The answer contains citations "
            "to pages that were not retrieved:",
            sorted(invalid_pages)
        )
        return False

    print("All cited pages are among the retrieved sources.")
    return True

In [87]:
validate_rag_answer(answer, sources)

False

In [88]:
from pathlib import Path

index_path = Path("data/processed/faiss_president_address")

index_path.parent.mkdir(parents=True, exist_ok=True)

vector_store.save_local(str(index_path))

print("FAISS index saved to:", index_path.resolve())

FAISS index saved to: C:\Users\Tony\Documents\Lok_Sabha_AI\data\processed\faiss_president_address


In [89]:
import re

def make_citation_safe(answer, docs):
    retrieved_pages = {
        int(doc.metadata["pdf_page"])
        for doc in docs
    }

    cited_pages = {
        int(page)
        for page in re.findall(
            r"\[PDF p\.\s*(\d+)\]",
            answer
        )
    }

    invalid_pages = cited_pages - retrieved_pages

    if invalid_pages:
        evidence = "\n\n".join(
            f"[PDF p. {doc.metadata['pdf_page']}]\n"
            f"{doc.page_content}"
            for doc in docs
        )

        return (
            "The generated answer failed citation validation. "
            f"Unverified page citations: {sorted(invalid_pages)}. "
            "Please review the retrieved evidence below instead.\n\n"
            + evidence
        )

    return answer

In [90]:
safe_answer = make_citation_safe(answer, sources)
print(safe_answer)

The generated answer failed citation validation. Unverified page citations: [64]. Please review the retrieved evidence below instead.

[PDF p. 80]
this. 
 
My government is engaged in making India a major space power in the 
world. It is a means to improve human life. Besides, this also attempts to increase 
India’s share in the space economy. Major decisions have been taken to expand 
India’s space programme. This has led to the formation of many new space 
startups. The day is not far when India’s Gaganyaan will reach space. 
Honorable Members, my government has made India one of the world’s

[PDF p. 66]
31.01.2024 
 
64 
 
has ensured participation of the private sector in the defence sector. Our 
government has opened up the space sector also for young start-ups. 
Honorable Members, my government acknowledges the contribution of 
wealth creators and believes in the capabilities of India’s private sector.We are 
committed to creating a conducive environment for doing business in Ind

In [93]:
from langchain_community.vectorstores import FAISS
from pathlib import Path

index_path = Path("data/processed/faiss_president_address")

loaded_vector_store = FAISS.load_local(
    str(index_path),
    embedding_model,
    allow_dangerous_deserialization=True
)

print("Reloaded chunks:", loaded_vector_store.index.ntotal)

Reloaded chunks: 103


In [94]:
question = "What did the President say about India's space programme?"

test_docs = loaded_vector_store.similarity_search(question, k=3)

for doc in test_docs:
    print(f"PDF page: {doc.metadata['pdf_page']}")
    print(doc.page_content[:250])
    print("-" * 40)

PDF page: 80
this. 
 
My government is engaged in making India a major space power in the 
world. It is a means to improve human life. Besides, this also attempts to increase 
India’s share in the space economy. Major decisions have been taken to expand 
India’s 
----------------------------------------
PDF page: 66
31.01.2024 
 
64 
 
has ensured participation of the private sector in the defence sector. Our 
government has opened up the space sector also for young start-ups. 
Honorable Members, my government acknowledges the contribution of 
wealth creators an
----------------------------------------
PDF page: 84
31.01.2024 
 
82 
 
- Middle East - Europe Corridor was announced. This corridor will further 
strengthen India’s maritime capability. The launch of the Global Biofuel Alliance is 
also a big event. Such steps are expanding India’s role in solving 
g
----------------------------------------


In [95]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template("""
You are a parliamentary research assistant.

Answer using only the supplied source passages.
Do not invent facts or add unsupported claims.
If the evidence is insufficient, say so.
Do not write page citations in the answer.
The application will list the retrieved source pages separately.

SOURCE PASSAGES:
{context}

QUESTION:
{question}

ANSWER:
""")

def ask_parliament(question, k=3):
    docs = loaded_vector_store.similarity_search(question, k=k)

    context = "\n\n".join(
        f"[PDF page {doc.metadata['pdf_page']}]\n{doc.page_content}"
        for doc in docs
    )

    response = llm.invoke(
        rag_prompt.format_messages(
            context=context,
            question=question
        )
    )

    source_pages = sorted({
        int(doc.metadata["pdf_page"])
        for doc in docs
    })

    print("Question:", question)
    print("\nAnswer:\n", response.content.strip())
    print(
        "\nSources:",
        ", ".join(f"PDF p. {page}" for page in source_pages)
    )

    return response.content.strip(), docs

In [96]:
answer, sources = ask_parliament(
    "What did the President say about India's space programme?"
)

Question: What did the President say about India's space programme?

Answer:
 The President mentioned that major decisions have been taken to expand India's space programme, which has led to the formation of many new space startups. They also mentioned that India's Gaganyaan will reach space, and that the government is engaged in making India a major space power in the world, which is a means to improve human life and increase India's share in the space economy.

Sources: PDF p. 66, PDF p. 80, PDF p. 84
